# Build a Simple RAG System

## Install OpenAI, and LangChain dependencies

In [1]:
!pip install langchain==1.2.12
!pip install langchain-openai==1.1.11
!pip install langchain-community==0.4.1
!pip install jq==1.11.0
!pip install pymupdf==1.27.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.8/773.8 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 57.4 MB/s eta 0:00:00


## Install Chroma Vector DB and LangChain wrapper

In [2]:
!pip install langchain-chroma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 1.2 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentelemetr

## Enter Open AI API Key

In [3]:
from getpass import getpass

OPENAI_KEY = getpass('Enter Open AI API Key: ')

Enter Open AI API Key: ··········


## Setup Environment Variables

In [4]:
import os

os.environ['OPENAI_API_KEY'] = OPENAI_KEY

### Open AI Embedding Models

LangChain enables us to access Open AI embedding models which include the newest models: a smaller and highly efficient `text-embedding-3-small` model, and a larger and more powerful `text-embedding-3-large` model.

In [5]:
from langchain_openai import OpenAIEmbeddings

# details here: https://openai.com/blog/new-embedding-models-and-api-updates
openai_embed_model = OpenAIEmbeddings(model='text-embedding-3-small')

## Loading and Processing the Data

### Get the dataset

In [6]:
# if you can't download using the following code
# go to https://drive.google.com/file/d/1aZxZejfteVuofISodUrY2CDoyuPLYDGZ download it
# manually upload it on colab
!gdown 1aZxZejfteVuofISodUrY2CDoyuPLYDGZ

Downloading...
From: https://drive.google.com/uc?id=1aZxZejfteVuofISodUrY2CDoyuPLYDGZ
To: /content/rag_docs.zip
100% 5.92M/5.92M [00:00<00:00, 30.8MB/s]


In [7]:
!unzip rag_docs.zip

Archive:  rag_docs.zip
   creating: rag_docs/
  inflating: rag_docs/attention_paper.pdf  
  inflating: rag_docs/cnn_paper.pdf  
  inflating: rag_docs/resnet_paper.pdf  
  inflating: rag_docs/vision_transformer.pdf  
  inflating: rag_docs/wikidata_rag_demo.jsonl  


### Load and Process JSON Documents

In [8]:
from langchain_community.document_loaders import JSONLoader

loader = JSONLoader(
    file_path="./rag_docs/wikidata_rag_demo.jsonl",
    jq_schema=".",
    text_content=False,
    json_lines=True
)
wiki_docs = loader.load()

In [9]:
len(wiki_docs)

1801

In [10]:
wiki_docs[3]

Document(metadata={'source': '/content/rag_docs/wikidata_rag_demo.jsonl', 'seq_num': 4}, page_content='{"id": "71548", "title": "Chi-square distribution", "paragraphs": ["In probability theory and statistics, the chi-square distribution (also chi-squared or formula_1\\u00a0 distribution) is one of the most widely used theoretical probability distributions. Chi-square distribution with formula_2 degrees of freedom is written as formula_3. It is a special case of gamma distribution.", "Chi-square distribution is primarily used in statistical significance tests and confidence intervals. It is useful, because it is relatively easy to show that certain probability distributions come close to it, under certain conditions. One of these conditions is that the null hypothesis must be true. Another one is that the different random variables (or observations) must be independent of each other."]}')

In [11]:
import json
from langchain_core.documents import Document

wiki_docs_processed = []

for doc in wiki_docs:
    doc = json.loads(doc.page_content)
    metadata = {
        "title": doc["title"],
        "id": doc["id"],
        "source": "Wikipedia"
    }
    data = " ".join(doc["paragraphs"])
    wiki_docs_processed.append(Document(page_content=data, metadata=metadata))

In [12]:
wiki_docs_processed[3]

Document(metadata={'title': 'Chi-square distribution', 'id': '71548', 'source': 'Wikipedia'}, page_content='In probability theory and statistics, the chi-square distribution (also chi-squared or formula_1\xa0 distribution) is one of the most widely used theoretical probability distributions. Chi-square distribution with formula_2 degrees of freedom is written as formula_3. It is a special case of gamma distribution. Chi-square distribution is primarily used in statistical significance tests and confidence intervals. It is useful, because it is relatively easy to show that certain probability distributions come close to it, under certain conditions. One of these conditions is that the null hypothesis must be true. Another one is that the different random variables (or observations) must be independent of each other.')

### Load and Process PDF documents

#### Create Simple Document Chunks for Standard Retrieval

Here we just use simple chunking where each chunk is a fixed size of <= 3500 characters and overlap of 200 characters for any small isolated chunks (you can and should experiment with various chunk sizes and overlaps)

In [13]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def create_simple_chunks(file_path, chunk_size=3500, chunk_overlap=0):

    print('Loading pages:', file_path)
    loader = PyMuPDFLoader(file_path)
    doc_pages = loader.load()

    print('Chunking pages:', file_path)
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    doc_chunks = splitter.split_documents(doc_pages)
    print('Finished processing:', file_path)
    print()
    return doc_chunks

In [14]:
from glob import glob

pdf_files = glob('./rag_docs/*.pdf')
pdf_files

['./rag_docs/resnet_paper.pdf',
 './rag_docs/attention_paper.pdf',
 './rag_docs/vision_transformer.pdf',
 './rag_docs/cnn_paper.pdf']

In [15]:
paper_docs = []
for fp in pdf_files:
    paper_docs.extend(create_simple_chunks(file_path=fp,
                                           chunk_size=3500,
                                           chunk_overlap=200))

Loading pages: ./rag_docs/resnet_paper.pdf
Chunking pages: ./rag_docs/resnet_paper.pdf
Finished processing: ./rag_docs/resnet_paper.pdf

Loading pages: ./rag_docs/attention_paper.pdf
Chunking pages: ./rag_docs/attention_paper.pdf
Finished processing: ./rag_docs/attention_paper.pdf

Loading pages: ./rag_docs/vision_transformer.pdf
Chunking pages: ./rag_docs/vision_transformer.pdf
Finished processing: ./rag_docs/vision_transformer.pdf

Loading pages: ./rag_docs/cnn_paper.pdf
Chunking pages: ./rag_docs/cnn_paper.pdf
Finished processing: ./rag_docs/cnn_paper.pdf



In [16]:
len(paper_docs)

79

In [17]:
paper_docs[0]

Document(metadata={'producer': 'pdfTeX-1.40.12', 'creator': 'LaTeX with hyperref package', 'creationdate': '2015-12-11T01:13:45+00:00', 'source': './rag_docs/resnet_paper.pdf', 'file_path': './rag_docs/resnet_paper.pdf', 'total_pages': 12, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2015-12-11T01:13:45+00:00', 'trapped': '', 'modDate': 'D:20151211011345Z', 'creationDate': 'D:20151211011345Z', 'page': 0}, page_content='Deep Residual Learning for Image Recognition\nKaiming He\nXiangyu Zhang\nShaoqing Ren\nJian Sun\nMicrosoft Research\n{kahe, v-xiangz, v-shren, jiansun}@microsoft.com\nAbstract\nDeeper neural networks are more difﬁcult to train. We\npresent a residual learning framework to ease the training\nof networks that are substantially deeper than those used\npreviously. We explicitly reformulate the layers as learn-\ning residual functions with reference to the layer inputs, in-\nstead of learning unreferenced functions. We provide com

### Combine all document chunks in one list

In [18]:
len(wiki_docs_processed)

1801

In [19]:
total_docs = wiki_docs_processed + paper_docs
len(total_docs)

1880

## Index Document Chunks and Embeddings in Vector DB

Here we initialize a connection to a Chroma vector DB client, and also we want to save to disk, so we simply initialize the Chroma client and pass the directory where we want the data to be saved to.

In [40]:
from langchain_chroma import Chroma

# create vector DB of docs and embeddings - takes < 30s on Colab
chroma_db = Chroma.from_documents(documents=total_docs,
                                  collection_name='my_db',
                                  embedding=openai_embed_model,
                                  # need to set the distance function to cosine else it uses euclidean by default
                                  # check https://docs.trychroma.com/guides#changing-the-distance-function
                                  collection_metadata={"hnsw:space": "cosine"},
                                  persist_directory="./my_db")

### Load Vector DB from disk

This is just to show once you have a vector database on disk you can just load and create a connection to it anytime

In [41]:
# load from disk
chroma_db = Chroma(persist_directory="./my_db",
                   collection_name='my_db',
                   embedding_function=openai_embed_model)

In [42]:
chroma_db

### Semantic Similarity based Retrieval

We use simple cosine similarity here and retrieve the top 5 similar documents based on the user input query

In [43]:
similarity_retriever = chroma_db.as_retriever(search_type="similarity",
                                              search_kwargs={"k": 5})

In [44]:
from IPython.display import display, Markdown

def display_docs(docs):
    for doc in docs:
        print('Metadata:', doc.metadata)
        print('Content Brief:')
        display(Markdown(doc.page_content[:1000]))
        print()

In [45]:
query = "what is machine learning?"
top_docs = similarity_retriever.invoke(query)
display_docs(top_docs)

Metadata: {'id': '564928', 'title': 'Machine learning', 'source': 'Wikipedia'}
Content Brief:


Machine learning gives computers the ability to learn without being explicitly programmed (Arthur Samuel, 1959). It is a subfield of computer science. The idea came from work in artificial intelligence. Machine learning explores the study and construction of algorithms which can learn and make predictions on data. Such algorithms follow programmed instructions, but can also make predictions or decisions based on data. They build a model from sample inputs. Machine learning is done where designing and programming explicit algorithms cannot be done. Examples include spam filtering, detection of network intruders or malicious insiders working towards a data breach, optical character recognition (OCR), search engines and computer vision.


Metadata: {'title': 'Machine learning', 'source': 'Wikipedia', 'id': '564928'}
Content Brief:


Machine learning gives computers the ability to learn without being explicitly programmed (Arthur Samuel, 1959). It is a subfield of computer science. The idea came from work in artificial intelligence. Machine learning explores the study and construction of algorithms which can learn and make predictions on data. Such algorithms follow programmed instructions, but can also make predictions or decisions based on data. They build a model from sample inputs. Machine learning is done where designing and programming explicit algorithms cannot be done. Examples include spam filtering, detection of network intruders or malicious insiders working towards a data breach, optical character recognition (OCR), search engines and computer vision.


Metadata: {'id': '359370', 'title': 'Supervised learning', 'source': 'Wikipedia'}
Content Brief:


In machine learning, supervised learning is the task of inferring a function from labelled training data. The results of the training are known beforehand, the system simply learns how to get to these results correctly. Usually, such systems work with vectors. They get the training data and the result of the training as two vectors and produce a "classifier". Usually, the system uses inductive reasoning to generalize the training data.


Metadata: {'title': 'Supervised learning', 'id': '359370', 'source': 'Wikipedia'}
Content Brief:


In machine learning, supervised learning is the task of inferring a function from labelled training data. The results of the training are known beforehand, the system simply learns how to get to these results correctly. Usually, such systems work with vectors. They get the training data and the result of the training as two vectors and produce a "classifier". Usually, the system uses inductive reasoning to generalize the training data.


Metadata: {'id': '663523', 'source': 'Wikipedia', 'title': 'Deep learning'}
Content Brief:


Deep learning (also called deep structured learning or hierarchical learning) is a kind of machine learning, which is mostly used with certain kinds of neural networks. As with other kinds of machine-learning, learning sessions can be unsupervised, semi-supervised, or supervised. In many cases, structures are organised so that there is at least one intermediate layer (or hidden layer), between the input layer and the output layer. Certain tasks, such as as recognizing and understanding speech, images or handwriting, is easy to do for humans. However, for a computer, these tasks are very difficult to do. In a multi-layer neural network (having more than two layers), the information processed will become more abstract with each added layer. Deep learning models are inspired by information processing and communication patterns in biological nervous systems; they are different from the structural and functional properties of biological brains (especially the human brain) in many ways, whic

In [46]:
query = "what is the difference between transformers and vision transformers?"
top_docs = similarity_retriever.invoke(query)
display_docs(top_docs)

Metadata: {'creator': 'LaTeX with hyperref', 'creationDate': 'D:20210604001958Z', 'modDate': 'D:20210604001958Z', 'producer': 'pdfTeX-1.40.21', 'title': '', 'moddate': '2021-06-04T00:19:58+00:00', 'format': 'PDF 1.5', 'file_path': './rag_docs/vision_transformer.pdf', 'total_pages': 22, 'subject': '', 'author': '', 'trapped': '', 'page': 7, 'keywords': '', 'source': './rag_docs/vision_transformer.pdf', 'creationdate': '2021-06-04T00:19:58+00:00'}
Content Brief:


Published as a conference paper at ICLR 2021
4.4
SCALING STUDY
We perform a controlled scaling study of different models by evaluating transfer performance from
JFT-300M. In this setting data size does not bottleneck the models’ performances, and we assess
performance versus pre-training cost of each model. The model set includes: 7 ResNets, R50x1,
R50x2 R101x1, R152x1, R152x2, pre-trained for 7 epochs, plus R152x2 and R200x3 pre-trained
for 14 epochs; 6 Vision Transformers, ViT-B/32, B/16, L/32, L/16, pre-trained for 7 epochs, plus
L/16 and H/14 pre-trained for 14 epochs; and 5 hybrids, R50+ViT-B/32, B/16, L/32, L/16 pre-
trained for 7 epochs, plus R50+ViT-L/16 pre-trained for 14 epochs (for hybrids, the number at the
end of the model name stands not for the patch size, but for the total dowsampling ratio in the ResNet
backbone).
Figure 5 contains the transfer performance versus total pre-training compute (see Appendix D.5
for details on computational costs). Detailed results per mode


Metadata: {'moddate': '2021-06-04T00:19:58+00:00', 'subject': '', 'total_pages': 22, 'creationDate': 'D:20210604001958Z', 'author': '', 'format': 'PDF 1.5', 'creator': 'LaTeX with hyperref', 'source': './rag_docs/vision_transformer.pdf', 'modDate': 'D:20210604001958Z', 'producer': 'pdfTeX-1.40.21', 'trapped': '', 'file_path': './rag_docs/vision_transformer.pdf', 'title': '', 'page': 7, 'keywords': '', 'creationdate': '2021-06-04T00:19:58+00:00'}
Content Brief:


Published as a conference paper at ICLR 2021
4.4
SCALING STUDY
We perform a controlled scaling study of different models by evaluating transfer performance from
JFT-300M. In this setting data size does not bottleneck the models’ performances, and we assess
performance versus pre-training cost of each model. The model set includes: 7 ResNets, R50x1,
R50x2 R101x1, R152x1, R152x2, pre-trained for 7 epochs, plus R152x2 and R200x3 pre-trained
for 14 epochs; 6 Vision Transformers, ViT-B/32, B/16, L/32, L/16, pre-trained for 7 epochs, plus
L/16 and H/14 pre-trained for 14 epochs; and 5 hybrids, R50+ViT-B/32, B/16, L/32, L/16 pre-
trained for 7 epochs, plus R50+ViT-L/16 pre-trained for 14 epochs (for hybrids, the number at the
end of the model name stands not for the patch size, but for the total dowsampling ratio in the ResNet
backbone).
Figure 5 contains the transfer performance versus total pre-training compute (see Appendix D.5
for details on computational costs). Detailed results per mode


Metadata: {'format': 'PDF 1.5', 'source': './rag_docs/vision_transformer.pdf', 'creator': 'LaTeX with hyperref', 'title': '', 'creationdate': '2021-06-04T00:19:58+00:00', 'file_path': './rag_docs/vision_transformer.pdf', 'keywords': '', 'page': 0, 'author': '', 'modDate': 'D:20210604001958Z', 'subject': '', 'producer': 'pdfTeX-1.40.21', 'total_pages': 22, 'moddate': '2021-06-04T00:19:58+00:00', 'creationDate': 'D:20210604001958Z', 'trapped': ''}
Content Brief:


Published as a conference paper at ICLR 2021
AN IMAGE IS WORTH 16X16 WORDS:
TRANSFORMERS FOR IMAGE RECOGNITION AT SCALE
Alexey Dosovitskiy∗,†, Lucas Beyer∗, Alexander Kolesnikov∗, Dirk Weissenborn∗,
Xiaohua Zhai∗, Thomas Unterthiner, Mostafa Dehghani, Matthias Minderer,
Georg Heigold, Sylvain Gelly, Jakob Uszkoreit, Neil Houlsby∗,†
∗equal technical contribution, †equal advising
Google Research, Brain Team
{adosovitskiy, neilhoulsby}@google.com
ABSTRACT
While the Transformer architecture has become the de-facto standard for natural
language processing tasks, its applications to computer vision remain limited. In
vision, attention is either applied in conjunction with convolutional networks, or
used to replace certain components of convolutional networks while keeping their
overall structure in place. We show that this reliance on CNNs is not necessary
and a pure transformer applied directly to sequences of image patches can perform
very well on image classiﬁcation tasks. When pre-traine


Metadata: {'author': '', 'modDate': 'D:20210604001958Z', 'moddate': '2021-06-04T00:19:58+00:00', 'total_pages': 22, 'subject': '', 'source': './rag_docs/vision_transformer.pdf', 'creationDate': 'D:20210604001958Z', 'creationdate': '2021-06-04T00:19:58+00:00', 'keywords': '', 'file_path': './rag_docs/vision_transformer.pdf', 'page': 0, 'title': '', 'format': 'PDF 1.5', 'producer': 'pdfTeX-1.40.21', 'trapped': '', 'creator': 'LaTeX with hyperref'}
Content Brief:


Published as a conference paper at ICLR 2021
AN IMAGE IS WORTH 16X16 WORDS:
TRANSFORMERS FOR IMAGE RECOGNITION AT SCALE
Alexey Dosovitskiy∗,†, Lucas Beyer∗, Alexander Kolesnikov∗, Dirk Weissenborn∗,
Xiaohua Zhai∗, Thomas Unterthiner, Mostafa Dehghani, Matthias Minderer,
Georg Heigold, Sylvain Gelly, Jakob Uszkoreit, Neil Houlsby∗,†
∗equal technical contribution, †equal advising
Google Research, Brain Team
{adosovitskiy, neilhoulsby}@google.com
ABSTRACT
While the Transformer architecture has become the de-facto standard for natural
language processing tasks, its applications to computer vision remain limited. In
vision, attention is either applied in conjunction with convolutional networks, or
used to replace certain components of convolutional networks while keeping their
overall structure in place. We show that this reliance on CNNs is not necessary
and a pure transformer applied directly to sequences of image patches can perform
very well on image classiﬁcation tasks. When pre-traine


Metadata: {'producer': 'pdfTeX-1.40.21', 'modDate': 'D:20210604001958Z', 'format': 'PDF 1.5', 'file_path': './rag_docs/vision_transformer.pdf', 'title': '', 'creationDate': 'D:20210604001958Z', 'subject': '', 'trapped': '', 'creationdate': '2021-06-04T00:19:58+00:00', 'creator': 'LaTeX with hyperref', 'moddate': '2021-06-04T00:19:58+00:00', 'source': './rag_docs/vision_transformer.pdf', 'total_pages': 22, 'keywords': '', 'author': '', 'page': 1}
Content Brief:


Published as a conference paper at ICLR 2021
inherent to CNNs, such as translation equivariance and locality, and therefore do not generalize well
when trained on insufﬁcient amounts of data.
However, the picture changes if the models are trained on larger datasets (14M-300M images). We
ﬁnd that large scale training trumps inductive bias. Our Vision Transformer (ViT) attains excellent
results when pre-trained at sufﬁcient scale and transferred to tasks with fewer datapoints. When
pre-trained on the public ImageNet-21k dataset or the in-house JFT-300M dataset, ViT approaches
or beats state of the art on multiple image recognition benchmarks. In particular, the best model
reaches the accuracy of 88.55% on ImageNet, 90.72% on ImageNet-ReaL, 94.55% on CIFAR-100,
and 77.63% on the VTAB suite of 19 tasks.
2
RELATED WORK
Transformers were proposed by Vaswani et al. (2017) for machine translation, and have since be-
come the state of the art method in many NLP tasks. Large Transformer-based mo

## Build the RAG Pipeline

In [47]:
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = """You are an assistant who is an expert in question-answering tasks.
                Answer the following question using only the following pieces of retrieved context.
                If the answer is not in the context, do not make up answers, just say that you don't know.
                Keep the answer detailed and well formatted based on the information from the context.

                Question:
                {question}

                Context:
                {context}

                Answer:
            """

rag_prompt_template = ChatPromptTemplate.from_template(rag_prompt)

In [48]:
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

chatgpt = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

qa_rag_chain = (
    {
        "context": (similarity_retriever
                      |
                    format_docs),
        "question": RunnablePassthrough()
    }
      |
    rag_prompt_template
      |
    chatgpt
)

In [49]:
from IPython.display import display, Markdown

query = "What is machine learning?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

Machine learning is a subfield of computer science that provides computers the ability to learn without being explicitly programmed, a concept first articulated by Arthur Samuel in 1959. It originated from research in artificial intelligence and focuses on the study and construction of algorithms that can learn from and make predictions based on data. 

These algorithms operate by following programmed instructions while also being capable of making predictions or decisions based on the data they process. They build models from sample inputs, which allows them to function in scenarios where explicit algorithm design and programming are not feasible. 

Examples of machine learning applications include:
- Spam filtering
- Detection of network intruders or malicious insiders
- Optical character recognition (OCR)
- Search engines
- Computer vision

Within machine learning, there are various approaches, including supervised learning, which involves inferring a function from labeled training data. In supervised learning, the outcomes of the training are known in advance, and the system learns to achieve these results correctly, typically using vectors to represent training data and outcomes.

Additionally, deep learning is a specialized area of machine learning that utilizes neural networks, often with multiple layers (known as deep structured learning). This approach is particularly effective for complex tasks such as speech recognition, image understanding, and handwriting recognition, which are challenging for computers but relatively easy for humans. Deep learning models are inspired by the information processing patterns of biological nervous systems, although they differ significantly from the structural and functional properties of biological brains.

In [51]:
query = "What is a CNN?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

A CNN, or Convolutional Neural Network, is a type of deep learning algorithm primarily used for processing structured grid data, such as images. CNNs are particularly effective in tasks like image recognition and classification due to their ability to automatically detect and learn features from the input data.

### Key Components of CNNs:
1. **Convolutional Layers**: These layers apply convolution operations to the input, allowing the network to learn spatial hierarchies of features. Each neuron in a convolutional layer is connected to a small region of the input, which helps in capturing local patterns.

2. **Pooling Layers**: Pooling layers reduce the spatial dimensions of the input volume, which helps in decreasing the computational load and controlling overfitting. Common pooling methods include max pooling and average pooling.

3. **Fully-Connected Layers**: After several convolutional and pooling layers, the high-level reasoning in the neural network is done through fully connected layers, where every neuron is connected to every neuron in the previous layer.

### Architecture:
- CNNs typically consist of a series of convolutional layers followed by pooling layers, and finally one or more fully connected layers. This architecture allows CNNs to learn complex features and patterns in the data with fewer parameters compared to traditional neural networks.

### Advantages:
- CNNs are known for their efficiency in processing large amounts of data and their ability to learn features automatically without the need for manual feature extraction. They are particularly powerful for image-related tasks due to their ability to capture spatial hierarchies.

### Resource Considerations:
- While CNNs are powerful, they can be resource-intensive, especially when dealing with large images. For instance, filtering a large image can result in a significant amount of memory usage, which necessitates careful management of computational resources.

In summary, CNNs are a specialized type of neural network designed to process data with a grid-like topology, making them highly effective for image analysis and other similar tasks.

In [52]:
query = "How is a resnet better than a CNN?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

A ResNet (Residual Network) is considered better than a traditional Convolutional Neural Network (CNN) for several reasons, primarily due to its architectural innovations that address common challenges faced by deeper networks. Here are the key advantages of ResNets over standard CNNs:

1. **Residual Learning**: ResNets introduce shortcut connections that allow gradients to flow through the network more effectively during training. This helps mitigate the vanishing gradient problem, which is common in deep networks. As a result, ResNets can be trained with greater depth without suffering from significant performance degradation.

2. **Improved Training Performance**: The architecture of ResNets enables them to achieve lower training errors compared to traditional CNNs. For instance, in experiments, a 34-layer ResNet outperformed an 18-layer ResNet by 2.8% in terms of accuracy, demonstrating that deeper networks can be effectively trained when using residual connections.

3. **Generalization Capability**: ResNets have shown to generalize better to unseen data. The ability to train deeper networks without overfitting allows them to perform better on validation and test datasets. For example, the 34-layer ResNet exhibited considerably lower training error and maintained competitive accuracy on validation sets.

4. **No Extra Parameters**: Despite being deeper, ResNets do not require additional parameters compared to their plain counterparts. This efficiency allows them to maintain a similar computational cost while achieving better performance.

5. **Performance Metrics**: In practical applications, ResNets have demonstrated superior performance metrics. For instance, a ResNet-101 achieved a top-5 classification error of 4.6% and a top-5 localization error of 14.4%, which are significantly lower than those of traditional CNNs.

In summary, the architectural innovations of ResNets, particularly the use of residual connections, allow them to train deeper networks effectively, improve generalization, and achieve better performance metrics compared to traditional CNNs.

In [53]:
query = "What is NLP and its relation to linguistics?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

Natural Language Processing (NLP) is a field within Artificial Intelligence that focuses on the interaction between computers and human languages. The primary goal of NLP is to enable computers to automatically understand, generate, and communicate in human languages. The term "Natural Language" specifically refers to human languages, distinguishing them from programming languages used in computing.

NLP is closely related to linguistics, which is the scientific study of language and its structure. This relationship is significant because understanding the nuances of human language—such as syntax, semantics, and context—is essential for developing effective NLP systems. By leveraging linguistic principles, NLP aims to improve how machines process and interpret human language, making it possible for them to perform tasks like translation, sentiment analysis, and conversational agents.

In summary, NLP is a crucial intersection of Artificial Intelligence and linguistics, aimed at programming computers to understand and produce human language effectively.

In [54]:
query = "What is the difference between AI, ML and DL?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

The differences between Artificial Intelligence (AI), Machine Learning (ML), and Deep Learning (DL) can be summarized as follows:

### 1. Artificial Intelligence (AI)
- **Definition**: AI refers to the ability of a computer program or machine to think and learn, mimicking human cognition. It encompasses a broad range of technologies and applications aimed at making machines "smart."
- **Scope**: AI is a field of study that includes various subfields, including machine learning and deep learning. It focuses on creating systems that can interpret external data, learn from it, and adapt to achieve specific goals.
- **Examples**: AI applications can range from simple rule-based systems to complex algorithms that can perform tasks like problem-solving and learning.

### 2. Machine Learning (ML)
- **Definition**: ML is a subfield of AI that gives computers the ability to learn from data without being explicitly programmed. It involves the study and construction of algorithms that can learn and make predictions based on input data.
- **Functionality**: ML algorithms build models from sample inputs and can make predictions or decisions based on data. They are particularly useful in scenarios where explicit programming is impractical.
- **Examples**: Common applications of ML include spam filtering, network intrusion detection, optical character recognition (OCR), and search engines.

### 3. Deep Learning (DL)
- **Definition**: DL is a specialized subset of machine learning that primarily uses neural networks with multiple layers (multi-layer neural networks) to process data. It is often referred to as deep structured learning or hierarchical learning.
- **Characteristics**: Deep learning models are designed to process information in a way that becomes increasingly abstract with each added layer. This makes them particularly effective for complex tasks such as speech recognition, image understanding, and handwriting recognition.
- **Inspiration**: DL models are inspired by the information processing patterns in biological nervous systems, although they differ significantly from the structural and functional properties of human brains.

### Summary
- **AI** is the overarching field that aims to create intelligent machines.
- **ML** is a subset of AI focused on algorithms that learn from data.
- **DL** is a further specialization within ML that utilizes deep neural networks for complex data processing tasks. 

In essence, while all three terms are interconnected, they represent different levels of complexity and specialization in the field of computer science.

In [55]:
query = "What is the difference between transformers and vision transformers?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

The difference between transformers and vision transformers primarily lies in their application and the way they process input data.

1. **Architecture and Input Processing**:
   - **Transformers**: Originally designed for natural language processing (NLP), transformers operate on sequences of tokens (words) and utilize self-attention mechanisms to capture relationships between these tokens. The input is typically a sequence of words, and the model learns to understand context and meaning through attention across the entire sequence.
   - **Vision Transformers (ViT)**: Vision transformers adapt the transformer architecture for image processing. Instead of processing sequences of words, ViTs treat images as sequences of patches. An image is divided into smaller patches, which are then flattened and linearly embedded into a lower-dimensional space. These embeddings are treated similarly to tokens in NLP, allowing the model to apply self-attention across the entire image.

2. **Inductive Bias**:
   - **Transformers**: In NLP, transformers benefit from the inherent structure of language, such as syntax and semantics, which helps in understanding context.
   - **Vision Transformers**: ViTs lack some of the inductive biases present in convolutional neural networks (CNNs), such as translation equivariance and locality. This means that while they can integrate information globally across the image, they may not generalize as well when trained on smaller datasets. However, when trained on large datasets, ViTs can outperform traditional CNNs.

3. **Performance and Efficiency**:
   - **Transformers**: They have become the standard in NLP due to their computational efficiency and scalability, allowing for the training of very large models.
   - **Vision Transformers**: ViTs have shown to require significantly less computational resources to achieve comparable or superior performance to state-of-the-art CNNs when pre-trained on large datasets. They can dominate the performance/compute trade-off, using approximately 2-4 times less compute to attain similar performance levels.

In summary, while both transformers and vision transformers utilize self-attention mechanisms, they differ in their input types (text vs. image patches), the inductive biases they leverage, and their performance characteristics based on the scale of training data.

In [56]:
query = "How is self-attention important in transformers?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

Self-attention is crucial in transformers for several reasons:

1. **Global Information Integration**: Self-attention allows the model to integrate information across the entire input, such as an image, even in the lowest layers of the network. This capability enables the model to attend to various parts of the input simultaneously, which is essential for understanding complex relationships and dependencies within the data.

2. **Attention Distance**: The concept of "attention distance" is significant in understanding how self-attention operates. It refers to the average distance in the input space across which information is integrated based on the attention weights. The findings indicate that some attention heads can attend to most of the image from the very first layers, demonstrating the model's ability to capture global context early in the processing.

3. **Localized vs. Global Attention**: While some attention heads exhibit highly localized attention (focusing on small regions of the input), others are capable of broader attention. This variation suggests that different heads can specialize in different types of information processing, which enhances the model's overall performance.

4. **Layer Depth and Attention**: The attention distance tends to increase with the depth of the network. As the model goes deeper, it can capture more extensive relationships within the data, which is beneficial for tasks that require a comprehensive understanding of the input.

5. **Semantic Relevance**: The model tends to focus on image regions that are semantically relevant for classification tasks. This indicates that self-attention not only facilitates the integration of information but also helps the model prioritize important features that contribute to its decision-making process.

In summary, self-attention is a fundamental mechanism in transformers that enables global information integration, allows for varying attention distances, and enhances the model's ability to focus on semantically relevant features, all of which contribute to the effectiveness of the transformer architecture in processing complex data.

In [57]:
query = "How does a resnet work?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

A ResNet, or Residual Network, operates by incorporating shortcut connections that allow the input to bypass one or more layers. This architecture is designed to address the optimization difficulties that arise when training very deep networks. Here’s a detailed breakdown of how a ResNet works based on the provided context:

### Key Features of ResNet:

1. **Shortcut Connections**:
   - ResNets introduce shortcut connections that enable the input to skip certain layers. This is particularly useful when the network depth increases, as it helps mitigate issues like vanishing gradients.
   - There are two types of shortcuts:
     - **Identity Mapping**: Used when the input and output dimensions are the same. This is a direct connection without any additional parameters.
     - **Projection Shortcut**: Employed when the dimensions change, typically using 1x1 convolutions to match the dimensions of the input and output.

2. **Layer Structure**:
   - The ResNet architecture consists of multiple layers of convolutional operations, typically using 3x3 filters. The layers are grouped into blocks, where each block contains a series of convolutions followed by a shortcut connection.
   - For example, a typical block might consist of two 3x3 convolutional layers followed by a shortcut that adds the input to the output of these layers.

3. **Training Dynamics**:
   - ResNets are trained using techniques such as batch normalization (BN) after each convolution and before activation, which helps stabilize the learning process.
   - The learning rate is often adjusted during training, starting at a higher value and decreasing as training progresses to ensure convergence.

4. **Performance**:
   - The introduction of residual connections allows deeper networks (e.g., 34-layer or 110-layer ResNets) to achieve better performance compared to their plain counterparts (networks without residual connections). This is evidenced by lower training errors and improved generalization capabilities.
   - In experiments, ResNets have shown to outperform plain networks of the same depth, demonstrating that they can effectively learn more complex representations without suffering from the degradation problem that typically affects deeper networks.

5. **Implementation**:
   - ResNets are implemented with careful consideration of data augmentation, weight initialization, and training schedules to optimize performance on tasks such as image classification.

In summary, ResNets leverage shortcut connections to facilitate the training of deep networks, allowing them to learn effectively and achieve higher accuracy compared to traditional architectures. This innovative approach addresses fundamental challenges in deep learning, particularly in optimizing very deep networks.

In [58]:
query = "What is LangGraph?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

I don't know.

In [59]:
query = "What is an Agentic AI System?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

The context provided does not contain specific information about an "Agentic AI System." Therefore, I don't know what an Agentic AI System is.

In [60]:
query = "What is LangChain?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

I don't know.